# Notebook de travail

Projet : Smart City Energy Forecasting — Tetouan


Importations et Chargement

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path

# Création des dossiers
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

# Chargement avec spécification stricte du format de la date
df = pd.read_csv(
    "../data/processed/tetouan_hourly_clean.csv", 
    index_col="datetime", 
    parse_dates=["datetime"],
    date_format="%Y-%m-%d %H:%M:%S"
)

print(f"Données chargées : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

Données chargées : (8736, 10)
Colonnes : ['temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows', 'zone1_power', 'zone2_power', 'zone3_power', 'target', 'total_load']


Variables Calendaires et Encodage Cyclique (Proxys Humains)

In [10]:
def add_time_features(data):
    df_feat = data.copy()
    
    if not isinstance(df_feat.index, pd.DatetimeIndex):
        df_feat.index = pd.to_datetime(df_feat.index)
        
    # Variables de base
    df_feat["hour"] = df_feat.index.hour
    df_feat["dayofweek"] = df_feat.index.dayofweek
    df_feat["month"] = df_feat.index.month
    
    # Indicateurs Smart City
    df_feat["is_weekend"] = (df_feat["dayofweek"] >= 5).astype(int)
    df_feat["is_morning_peak"] = df_feat["hour"].between(7, 9).astype(int)
    df_feat["is_evening_peak"] = df_feat["hour"].between(18, 21).astype(int)
    df_feat["is_offpeak"] = df_feat["hour"].between(0, 5).astype(int)
    
    # Encodage cyclique (L'heure 23 est proche de l'heure 0)
    df_feat["hour_sin"] = np.sin(2 * np.pi * df_feat["hour"] / 24)
    df_feat["hour_cos"] = np.cos(2 * np.pi * df_feat["hour"] / 24)
    
    return df_feat

df_features = add_time_features(df)
print("Création des variables temporelles réussie !")
display(df_features.head())

Création des variables temporelles réussie !


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load,hour,dayofweek,month,is_weekend,is_morning_peak,is_evening_peak,is_offpeak,hour_sin,hour_cos
datetime,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.196833,75.066667,0.081833,0.063500,0.098833,29197.974683,18026.747720,19252.048193,29197.974683,66476.770597,0,6,1,1,0,0,1,0.000000,1.000000
2017-01-01 01:00:00,5.548833,77.583333,0.082000,0.056833,0.112500,24657.215190,16078.419453,17042.891567,24657.215190,57778.526210,1,6,1,1,0,0,1,0.258819,0.965926
2017-01-01 02:00:00,5.054333,78.933333,0.082333,0.063000,0.129167,22083.037973,14330.699088,15676.144578,22083.037973,52089.881640,2,6,1,1,0,0,1,0.500000,0.866025
2017-01-01 03:00:00,5.004333,77.083333,0.082833,0.059833,0.141000,20811.139240,13219.452887,14883.855422,20811.139240,48914.447548,3,6,1,1,0,0,1,0.707107,0.707107
2017-01-01 04:00:00,5.097667,74.050000,0.082333,0.058000,0.122833,20475.949367,12921.580547,14317.108433,20475.949367,47714.638347,4,6,1,1,0,0,1,0.866025,0.500000


Variables Météorologiques et Lags (Mémoire de la série)

In [14]:
def add_weather_lags(data, target_col='target'):
    df_feat = data.copy()

    # SÉCURITÉ : Recréation de la cible (conservé comme garde-fou)
    if target_col not in df_feat.columns:
        print(f" Colonne '{target_col}' introuvable. Recréation à partir de 'zone1_power'...")
        df_feat[target_col] = df_feat['zone1_power']

    # Format explicite pour éviter le UserWarning si on force l'index
    if not isinstance(df_feat.index, pd.DatetimeIndex):
        df_feat.index = pd.to_datetime(df_feat.index, format="%Y-%m-%d %H:%M:%S", errors="coerce")

    # Degrés-jours de température (base 18°C)
    base_temp = 18.0
    df_feat['HDD'] = np.maximum(base_temp - df_feat['temperature'], 0)
    df_feat['CDD'] = np.maximum(df_feat['temperature'] - base_temp, 0)

    # Lags (Retards) dictés par l'ACF/PACF : 1h, 24h (la veille), 168h (la semaine passée)
    lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
    for lag in lags:
        df_feat[f'{target_col}_lag_{lag}'] = df_feat[target_col].shift(lag)

    # Statistiques glissantes (Rolling) basées sur le passé uniquement (shift 1)
    shifted_target = df_feat[target_col].shift(1)
    windows = [6, 12, 24]
    for w in windows:
        df_feat[f"{target_col}_rolling_mean_{w}"] = shifted_target.rolling(window=w).mean()
        
    return df_feat

# Application de la fonction
df_features = add_weather_lags(df_features)

# Suppression des lignes contenant des NaN générées par les Lags
df_features = df_features.dropna()
print(f"Dimensions après Feature Engineering : {df_features.shape}")
print(f"Lignes supprimées par dropna (lag_168) : {8736 - len(df_features)}")


Dimensions après Feature Engineering : (8400, 33)
Lignes supprimées par dropna (lag_168) : 336


Split Temporel et Normalisation Stricte (Data Leakage Prevention)

In [12]:
# Split Chronologique (70% Train, 15% Val, 15% Test) - JAMAIS d'aléatoire en séries temporelles
n = len(df_features)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df_features.iloc[:train_end].copy()
val = df_features.iloc[train_end:val_end].copy()
test = df_features.iloc[val_end:].copy()

# Normalisation 
features_to_scale = [col for col in train.columns if col not in ['target', 'zone1_power', 'zone2_power', 'zone3_power']]

scaler_x = StandardScaler()
train[features_to_scale] = scaler_x.fit_transform(train[features_to_scale])
val[features_to_scale] = scaler_x.transform(val[features_to_scale])
test[features_to_scale] = scaler_x.transform(test[features_to_scale])

# Sauvegarde des données prétraitées
train.to_csv("../data/processed/train.csv")
val.to_csv("../data/processed/val.csv")
test.to_csv("../data/processed/test.csv")
joblib.dump(scaler_x, "../models/scaler_x.pkl")

print("Pipeline de Feature Engineering et Split terminé avec succès.")

Pipeline de Feature Engineering et Split terminé avec succès.
